In [1]:
import re
import numpy as np
import pandas as pd
 
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [2]:
# need these for stopwords and sentiment later
nltk.download('stopwords')
nltk.download('vader_lexicon')
 
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rohit\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\rohit\AppData\Roaming\nltk_data...


In [3]:
df = pd.read_csv('frd.csv')
df.shape
 
df['label'].value_counts()

label
CG    20216
OR    20216
Name: count, dtype: int64

In [4]:
df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [5]:
# function to clean the review text before doing anything else
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)  # removing numbers/punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [6]:
# removing stopwords + stemming so the model doesn't get confused by common words
def preprocess(text):
    text = clean_text(text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words and len(w) > 1]
    tokens = [stemmer.stem(w) for w in tokens]
    return ' '.join(tokens)
 
df['clean_text'] = df['text_'].apply(preprocess)

In [7]:
# just checking one example to see if preprocessing worked fine
df['text_'].iloc[0]
 
df['clean_text'].iloc[0]
 
# converting labels to 0/1 since models need numbers not strings
# CG = fake review, OR = real review
df['label_enc'] = df['label'].map({'CG': 1, 'OR': 0})
 
# splitting into train and test, keeping 20% for testing
X_train_text, X_test_text, y_train, y_test, rating_train, rating_test = train_test_split(
    df['clean_text'], df['label_enc'], df['rating'],
    test_size=0.2, random_state=42, stratify=df['label_enc']
)
 
# saving the original (uncleaned) text too, will need it for the custom classifier later
raw_train_text = df.loc[X_train_text.index, 'text_']
raw_test_text = df.loc[X_test_text.index, 'text_']

In [8]:
# converting text into TFIDF vectors so the models can actually use it
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3)
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)
 
X_train_tfidf.shape
 
# trying out a bunch of models to see which one works best
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=30, n_jobs=-1, random_state=42),
    'Linear SVM': LinearSVC(random_state=42, max_iter=3000),
    'SGD Classifier': SGDClassifier(loss='log_loss', random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(),
}
 
results = {}
 
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
 
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    cm = confusion_matrix(y_test, preds)
 
    results[name] = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'confusion_matrix': cm}
 
    print(name)
    print('accuracy :', round(acc, 4))
    print('precision:', round(prec, 4))
    print('recall   :', round(rec, 4))
    print('f1 score :', round(f1, 4))
    print('confusion matrix:')
    print(cm)
    print()

Logistic Regression
accuracy : 0.8916
precision: 0.894
recall   : 0.8884
f1 score : 0.8912
confusion matrix:
[[3618  426]
 [ 451 3592]]

Random Forest
accuracy : 0.8174
precision: 0.8488
recall   : 0.7722
f1 score : 0.8087
confusion matrix:
[[3488  556]
 [ 921 3122]]

Linear SVM
accuracy : 0.8965
precision: 0.896
recall   : 0.8971
f1 score : 0.8966
confusion matrix:
[[3623  421]
 [ 416 3627]]

SGD Classifier
accuracy : 0.881
precision: 0.8837
recall   : 0.8776
f1 score : 0.8806
confusion matrix:
[[3577  467]
 [ 495 3548]]

Multinomial Naive Bayes
accuracy : 0.863
precision: 0.8692
recall   : 0.8546
f1 score : 0.8618
confusion matrix:
[[3524  520]
 [ 588 3455]]



In [9]:
# checking which model actually performed the best
best_model_name = max(results, key=lambda name: results[name]['accuracy'])
print('best model:', best_model_name, '-', results[best_model_name]['accuracy'])

best model: Linear SVM - 0.8965005564486213


In [10]:
# now trying my own custom classifier instead of just TFIDF
# calling it WESC - Weighted Emotion Sentiment Consistency
# idea: instead of using word frequencies, build features based
# on sentiment, rating vs sentiment mismatch, writing style etc
# and see if that alone can classify reviews well
# ------------------------------------------------------------
 
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.preprocessing import StandardScaler
 
sia = SentimentIntensityAnalyzer()

In [11]:
# words that show up a lot in fake/hype reviews, noticed this while checking the data
superlative_words = {
    'amazing', 'best', 'perfect', 'love', 'excellent', 'awesome',
    'great', 'wonderful', 'fantastic', 'incredible', 'outstanding',
    'flawless', 'superb', 'brilliant', 'exceptional'
}
def extract_wesc_features(texts, ratings):
    rows = []
    for text, rating in zip(texts, ratings):
        text = str(text)
        words = re.findall(r'\b\w+\b', text.lower())
        n_words = max(len(words), 1)
 
        sentiment = sia.polarity_scores(text)['compound']
 
        # checking if the star rating actually matches the tone of the text
        # e.g. 5 stars but text sounds neutral/negative = suspicious
        rating_scaled = (float(rating) - 3.0) / 2.0
        rating_sentiment_gap = abs(rating_scaled - sentiment)
 
        review_length = len(words)
        exclamation_density = (text.count('!') / max(len(text), 1)) * 100
        superlative_density = (sum(1 for w in words if w in superlative_words) / n_words) * 100
        lexical_diversity = len(set(words)) / n_words
        avg_word_length = np.mean([len(w) for w in words]) if words else 0
        punctuation_density = (len(re.findall(r'[.,!?;:]', text)) / max(len(text), 1)) * 100
        uppercase_ratio = sum(1 for c in text if c.isupper()) / max(len(text), 1)
        avg_sentence_length = n_words / max(text.count('.') + text.count('!') + text.count('?'), 1)
 
        rows.append({
            'sentiment_polarity': sentiment,
            'rating_sentiment_gap': rating_sentiment_gap,
            'review_length': review_length,
            'exclamation_density': exclamation_density,
            'superlative_density': superlative_density,
            'lexical_diversity': lexical_diversity,
            'avg_word_length': avg_word_length,
            'punctuation_density': punctuation_density,
            'uppercase_ratio': uppercase_ratio,
            'avg_sentence_length': avg_sentence_length,
        })
    return pd.DataFrame(rows)
 
X_train_wesc = extract_wesc_features(raw_train_text.tolist(), rating_train.tolist())
X_test_wesc = extract_wesc_features(raw_test_text.tolist(), rating_test.tolist())

In [12]:
# scaling since these features are on totally different scales (e.g. length vs ratio)
scaler = StandardScaler()
X_train_wesc_scaled = scaler.fit_transform(X_train_wesc)
X_test_wesc_scaled = scaler.transform(X_test_wesc)
 
wesc_model = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_leaf=5,
    class_weight='balanced', n_jobs=-1, random_state=42
)
wesc_model.fit(X_train_wesc_scaled, y_train)
wesc_preds = wesc_model.predict(X_test_wesc_scaled)
 
wesc_acc = accuracy_score(y_test, wesc_preds)
wesc_prec = precision_score(y_test, wesc_preds)
wesc_rec = recall_score(y_test, wesc_preds)
wesc_f1 = f1_score(y_test, wesc_preds)
 
print('WESC classifier results')
print('accuracy :', round(wesc_acc, 4))
print('precision:', round(wesc_prec, 4))
print('recall   :', round(wesc_rec, 4))
print('f1 score :', round(wesc_f1, 4))
 
# checking which features actually mattered the most for this model
importances = sorted(zip(X_train_wesc.columns, wesc_model.feature_importances_), key=lambda x: -x[1])
print('\nfeature importance:')
for feat, score in importances:
    print(feat, '-', round(score, 4))

WESC classifier results
accuracy : 0.8416
precision: 0.8385
recall   : 0.8462
f1 score : 0.8423

feature importance:
avg_word_length - 0.2716
lexical_diversity - 0.2317
review_length - 0.203
uppercase_ratio - 0.0634
avg_sentence_length - 0.06
sentiment_polarity - 0.0416
superlative_density - 0.0387
punctuation_density - 0.0364
exclamation_density - 0.027
rating_sentiment_gap - 0.0265


In [13]:
import joblib
import os

os.makedirs('models', exist_ok=True)

joblib.dump(vectorizer, 'models/tfidf_vectorizer.pkl')
joblib.dump(models['Linear SVM'], 'models/best_model.pkl')   # your best performing one
joblib.dump(scaler, 'models/wesc_scaler.pkl')
joblib.dump(wesc_model, 'models/wesc_model.pkl')

['models/wesc_model.pkl']